# NM i AI 2026 — Oppgave 1: Final Submission

**Strategi:** YOLOv8x → ONNX export (bevist å virke) + conf-tuning på val

Hva vi vet fungerer fra 0.8654-submission:
- ONNX export (unngår torch-versjonsmismatch)
- Enkel inference med augment=True
- imgsz=960 i inference

Nye forbedringer:
1. **YOLOv8x** (68M params vs 25M for YOLOv8m)
2. **imgsz=1280** i trening (fanger små produkter bedre)
3. **Produktbilder** (maks 4 per klasse — unngår overfitting til hvit bakgrunn)
4. **Conf-tuning** på val-sett (søker optimalt threshold)
5. **100%-retrain** etter valideringsgodkjenning

Score = 0.7 × detection_mAP@0.5 + 0.3 × classification_mAP

In [ ]:
# CELLE 1: Setup — installer ultralytics 8.1.0 (matcher sandbox), sjekk GPU
!pip install -q ultralytics==8.1.0
!pip uninstall -y wandb 2>/dev/null
!pip show ultralytics | grep Version

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Ingen GPU! Runtime → Change runtime type → T4/A100 GPU')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')

In [ ]:
# CELLE 2: Mount Drive og pakk ut data
from google.colab import drive
import zipfile, json
from pathlib import Path

drive.mount('/content/drive', force_remount=True)

print('Pakker ut train.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/train.zip', 'r') as z:
    z.extractall('/content/')

print('Pakker ut product_images.zip...')
with zipfile.ZipFile('/content/drive/MyDrive/product_images.zip', 'r') as z:
    z.extractall('/content/')

with open('/content/train/annotations.json') as f:
    coco = json.load(f)

print(f'Bilder:        {len(coco["images"])}')
print(f'Kategorier:    {len(coco["categories"])}')
print(f'Annotasjoner:  {len(coco["annotations"])}')
print(f'Kat-ID range:  {min(c["id"] for c in coco["categories"])} – {max(c["id"] for c in coco["categories"])}')

In [ ]:
# CELLE 3: Bygg YOLO-dataset med 85/15 split + produktbilder (maks 4/klasse)
import json, random, shutil
from pathlib import Path
from PIL import Image, ImageOps

MAX_PROD_PER_CLASS = 4   # Begrenser overfit til hvit bakgrunn
TRAIN_FRAC = 0.85
random.seed(42)

with open('/content/train/annotations.json') as f:
    coco = json.load(f)
with open('/content/NM_NGD_product_images/metadata.json') as f:
    meta = json.load(f)

cats_sorted = sorted(coco['categories'], key=lambda c: c['id'])
nc = len(cats_sorted)
print(f'nc={nc}')

# 85/15 split
imgs = coco['images'][:]
random.shuffle(imgs)
n_train = int(len(imgs) * TRAIN_FRAC)
train_imgs, val_imgs = imgs[:n_train], imgs[n_train:]
val_ids = {img['id'] for img in val_imgs}
print(f'Split: {len(train_imgs)} train / {len(val_imgs)} val')

val_anns = [a for a in coco['annotations'] if a['image_id'] in val_ids]
with open('/content/val_images.json', 'w') as f:
    json.dump(val_imgs, f)
with open('/content/val_annotations.json', 'w') as f:
    json.dump(val_anns, f)

# Mappestruktur
yolo = Path('/content/yolo_dataset')
for split in ['train', 'val']:
    (yolo / 'images' / split).mkdir(parents=True, exist_ok=True)
    (yolo / 'labels' / split).mkdir(parents=True, exist_ok=True)

ann_by_img = {}
for ann in coco['annotations']:
    ann_by_img.setdefault(ann['image_id'], []).append(ann)

def write_shelf(img_list, split):
    for img_info in img_list:
        fname = img_info['file_name']
        src = Path('/content/train/images') / fname
        shutil.copy(str(src), str(yolo / 'images' / split / fname))
        W = img_info.get('width') or Image.open(str(src)).width
        H = img_info.get('height') or Image.open(str(src)).height
        lines = []
        for ann in ann_by_img.get(img_info['id'], []):
            x, y, w, h = ann['bbox']
            cx = max(0.0, min(1.0, (x + w/2) / W))
            cy = max(0.0, min(1.0, (y + h/2) / H))
            nw = max(0.0, min(1.0, w / W))
            nh = max(0.0, min(1.0, h / H))
            if nw > 0 and nh > 0:
                lines.append(f'{ann["category_id"]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        (yolo / 'labels' / split / (Path(fname).stem + '.txt')).write_text('\n'.join(lines))

write_shelf(train_imgs, 'train')
write_shelf(val_imgs, 'val')
print(f'Hyllebilder skrevet.')

# Produktbilder med padding, maks MAX_PROD_PER_CLASS per klasse
cat_name_to_id = {c['name'].strip().upper(): c['id'] for c in coco['categories']}
code_to_cat = {}
for p in meta['products']:
    name = p['product_name'].strip().upper()
    if name in cat_name_to_id:
        code_to_cat[p['product_code']] = cat_name_to_id[name]

n_prod = 0
class_count = {}  # Teller antall produktbilder per klasse
img_base = Path('/content/NM_NGD_product_images')

for product_code, cat_id in code_to_cat.items():
    product_dir = img_base / product_code
    if not product_dir.exists():
        continue
    for img_path in sorted(product_dir.glob('*.jpg')):
        if class_count.get(cat_id, 0) >= MAX_PROD_PER_CLASS:
            break  # Nok bilder for denne klassen
        unique_name = f'prod_{product_code}_{img_path.stem}'
        dst_img = yolo / 'images' / 'train' / f'{unique_name}.jpg'
        dst_lbl = yolo / 'labels' / 'train' / f'{unique_name}.txt'

        img = Image.open(str(img_path)).convert('RGB')
        w, h = img.size
        # Padding 30-50% per side — realistisk bbox, ikke 1.0x1.0
        pad_frac = random.uniform(0.30, 0.50)
        pad_x, pad_y = int(w * pad_frac), int(h * pad_frac)
        bg = tuple(random.randint(170, 240) for _ in range(3))
        padded = ImageOps.expand(img, border=(pad_x, pad_y, pad_x, pad_y), fill=bg)
        padded.save(str(dst_img), quality=90)

        new_w, new_h = w + 2*pad_x, h + 2*pad_y
        cx = (pad_x + w/2) / new_w
        cy = (pad_y + h/2) / new_h
        dst_lbl.write_text(f'{cat_id} {cx:.6f} {cy:.6f} {w/new_w:.6f} {h/new_h:.6f}')
        class_count[cat_id] = class_count.get(cat_id, 0) + 1
        n_prod += 1

print(f'Produktbilder: {n_prod} totalt ({len(class_count)} unike klasser)')
print(f'Totalt train: {len(list((yolo/"images"/"train").iterdir()))} bilder')

# data.yaml
names_lines = [f"  {c['id']}: '{c['name'].replace(chr(39), '').replace(chr(34), '')}'"
               for c in cats_sorted]
(yolo / 'data.yaml').write_text(
    f'path: /content/yolo_dataset\ntrain: images/train\nval: images/val\nnc: {nc}\nnames:\n'
    + '\n'.join(names_lines)
)
print(f'data.yaml OK  nc={nc}')

In [ ]:
# CELLE 4: Tren YOLOv8x
import os, torch, functools, shutil
os.environ['WANDB_DISABLED'] = 'true'
torch.load = functools.partial(torch.load, weights_only=False)

from ultralytics import YOLO

# Juster batch etter GPU:
# A100 (40GB): batch=8  |  T4 (16GB): batch=4
BATCH = 8 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 4
print(f'Bruker batch={BATCH}')

model = YOLO('yolov8x.pt')
results = model.train(
    data='/content/yolo_dataset/data.yaml',
    epochs=150,
    imgsz=1280,
    batch=BATCH,
    patience=30,
    device=0,
    project='/content/runs',
    name='yolov8x_final',
    exist_ok=True,
    save=True,
    amp=True,
    workers=2,
    # Augmentering — balansert for hyllebilder
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    degrees=3.0,       # Lav rotasjon (produkter på hyller er nesten aldri rotert)
    translate=0.1,
    scale=0.5,
    mosaic=1.0,
    mixup=0.05,        # Lavt mixup — kan forvirre klassifisering av 356 klasser
    copy_paste=0.3,    # Bra for tette hyller
    erasing=0.2,
    fliplr=0.5,
    close_mosaic=15,
    # LR
    optimizer='AdamW',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    weight_decay=0.0005,
)

best_pt = '/content/runs/yolov8x_final/weights/best.pt'
shutil.copy(best_pt, '/content/drive/MyDrive/yolov8x_final_best.pt')
val_map = results.results_dict.get('metrics/mAP50(B)', '?')
print(f'Ferdig! Val mAP@0.5: {val_map}')
print('Lagret til Drive: yolov8x_final_best.pt')

In [ ]:
# CELLE 5: Finn optimalt conf-threshold på val-sett
# Søker over conf ∈ [0.05, 0.10, 0.15, 0.20, 0.25] og velger beste kombinert score
import json, torch, functools
from collections import defaultdict
from pathlib import Path

torch.load = functools.partial(torch.load, weights_only=False)

from ultralytics import YOLO

with open('/content/val_images.json') as f:
    val_imgs = json.load(f)
with open('/content/val_annotations.json') as f:
    val_anns = json.load(f)

val_input = Path('/content/val_input')
val_input.mkdir(exist_ok=True)
import shutil
for img_info in val_imgs:
    src = Path('/content/train/images') / img_info['file_name']
    if src.exists():
        shutil.copy(str(src), str(val_input / img_info['file_name']))

model = YOLO('/content/runs/yolov8x_final/weights/best.pt')
img_paths = sorted([p for p in val_input.iterdir()
                    if p.suffix.lower() in ('.jpg', '.jpeg', '.png')])
print(f'Val-bilder: {len(img_paths)}')

def iou_fn(a, b):
    ax2, ay2 = a[0]+a[2], a[1]+a[3]
    bx2, by2 = b[0]+b[2], b[1]+b[3]
    iw = max(0, min(ax2, bx2) - max(a[0], b[0]))
    ih = max(0, min(ay2, by2) - max(a[1], b[1]))
    inter = iw * ih
    union = a[2]*a[3] + b[2]*b[3] - inter
    return inter / union if union > 0 else 0.0

def compute_map(preds, gts, check_category=False):
    cats = set(g['category_id'] for g in gts) if check_category else {0}
    pb, gb = defaultdict(list), defaultdict(list)
    for p in preds:
        pb[p['category_id'] if check_category else 0].append(p)
    for g in gts:
        gb[g['category_id'] if check_category else 0].append(g)
    aps = []
    for cat in cats:
        cp = sorted(pb.get(cat, []), key=lambda x: -x['score'])
        cg = gb.get(cat, [])
        n = len(cg)
        if n == 0:
            continue
        matched = defaultdict(set)
        tps, fps = [], []
        for p in cp:
            ig = [g for g in (cg if check_category else gts)
                  if g['image_id'] == p['image_id']]
            bi, bx = 0, -1
            for gi, g in enumerate(ig):
                v = iou_fn(p['bbox'], g['bbox'])
                if v > bi:
                    bi, bx = v, gi
            if bi >= 0.5 and bx not in matched[p['image_id']]:
                matched[p['image_id']].add(bx)
                tps.append(1); fps.append(0)
            else:
                tps.append(0); fps.append(1)
        tc = [sum(tps[:i+1]) for i in range(len(tps))]
        fc = [sum(fps[:i+1]) for i in range(len(fps))]
        prec = [t/(t+f) if t+f > 0 else 0 for t, f in zip(tc, fc)]
        rec  = [t/n for t in tc]
        ap = sum(max([p for p, r in zip(prec, rec) if r >= t] or [0])
                 for t in [i/10 for i in range(11)]) / 11
        aps.append(ap)
    return sum(aps) / len(aps) if aps else 0.0

# Kjør inference én gang med lav conf, filtrer etterpå
print('Kjører inference med conf=0.01 (filtrer i etterkant)...')
all_preds = []
for img_path in img_paths:
    image_id = int(img_path.stem.split('_')[-1])
    results = model.predict(
        str(img_path), device=0, conf=0.01, iou=0.45,
        verbose=False, imgsz=960, augment=True,
    )
    for result in results:
        if result.boxes is None:
            continue
        for i in range(len(result.boxes)):
            x1, y1, x2, y2 = result.boxes.xyxy[i].tolist()
            all_preds.append({
                'image_id': image_id,
                'category_id': int(result.boxes.cls[i].item()),
                'bbox': [round(x1,1), round(y1,1), round(x2-x1,1), round(y2-y1,1)],
                'score': float(result.boxes.conf[i].item()),
            })

print(f'Totalt prediksjoner (conf>0.01): {len(all_preds)}')

# Søk over conf-thresholds
THRESHOLDS = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
best_conf, best_score = 0.15, 0.0
print(f'\n{"Conf":>6}  {"Det":>7}  {"Cls":>7}  {"Combined":>9}')
print('-' * 40)
for conf in THRESHOLDS:
    preds = [p for p in all_preds if p['score'] >= conf]
    det = compute_map(preds, val_anns, check_category=False)
    cls = compute_map(preds, val_anns, check_category=True)
    comb = 0.7*det + 0.3*cls
    marker = ' <-- best' if comb > best_score else ''
    print(f'{conf:>6.2f}  {det:>7.4f}  {cls:>7.4f}  {comb:>9.4f}{marker}')
    if comb > best_score:
        best_score, best_conf = comb, conf

print(f'\nBeste conf: {best_conf}  →  score {best_score:.4f}')
print(f'Vår beste submission: 0.8654')

with open('/content/val_score.json', 'w') as f:
    json.dump({'combined': best_score, 'best_conf': best_conf, 'beste': 0.8654}, f)
print('Lagret val_score.json')

In [ ]:
# CELLE 6: (Valgfri) Retren på 100% data hvis val-score > 0.8654
import json
with open('/content/val_score.json') as f:
    vs = json.load(f)

if vs['combined'] <= vs['beste']:
    print(f"Score {vs['combined']:.4f} <= beste {vs['beste']}. Hopper over 100%-retrain.")
    print('Gå til celle 7 og submit med 85%-modellen likevel.')
else:
    print(f"Score {vs['combined']:.4f} > {vs['beste']}. Retrener på 100%!")

    import json, random, shutil, os, torch, functools
    from pathlib import Path
    from PIL import Image, ImageOps

    torch.load = functools.partial(torch.load, weights_only=False)
    os.environ['WANDB_DISABLED'] = 'true'

    with open('/content/train/annotations.json') as f:
        coco = json.load(f)
    with open('/content/NM_NGD_product_images/metadata.json') as f:
        meta = json.load(f)

    MAX_PROD_PER_CLASS = 4
    random.seed(42)

    yolo100 = Path('/content/yolo_100pct')
    (yolo100 / 'images' / 'train').mkdir(parents=True, exist_ok=True)
    (yolo100 / 'labels' / 'train').mkdir(parents=True, exist_ok=True)

    ann_by_img = {}
    for ann in coco['annotations']:
        ann_by_img.setdefault(ann['image_id'], []).append(ann)

    for img_info in coco['images']:
        fname = img_info['file_name']
        shutil.copy(str(Path('/content/train/images') / fname),
                    str(yolo100 / 'images' / 'train' / fname))
        W = img_info.get('width', 2000)
        H = img_info.get('height', 1500)
        lines = []
        for ann in ann_by_img.get(img_info['id'], []):
            x, y, w, h = ann['bbox']
            cx = max(0.0, min(1.0, (x + w/2) / W))
            cy = max(0.0, min(1.0, (y + h/2) / H))
            nw = max(0.0, min(1.0, w / W))
            nh = max(0.0, min(1.0, h / H))
            if nw > 0 and nh > 0:
                lines.append(f'{ann["category_id"]} {cx:.6f} {cy:.6f} {nw:.6f} {nh:.6f}')
        (yolo100 / 'labels' / 'train' / (Path(fname).stem + '.txt')).write_text('\n'.join(lines))

    cat_name_to_id = {c['name'].strip().upper(): c['id'] for c in coco['categories']}
    code_to_cat = {}
    for p in meta['products']:
        name = p['product_name'].strip().upper()
        if name in cat_name_to_id:
            code_to_cat[p['product_code']] = cat_name_to_id[name]

    n_prod, class_count = 0, {}
    img_base = Path('/content/NM_NGD_product_images')
    for pc, cat_id in code_to_cat.items():
        pd_dir = img_base / pc
        if not pd_dir.exists(): continue
        for ip in sorted(pd_dir.glob('*.jpg')):
            if class_count.get(cat_id, 0) >= MAX_PROD_PER_CLASS: break
            un = f'prod_{pc}_{ip.stem}'
            dst_img = yolo100 / 'images' / 'train' / f'{un}.jpg'
            dst_lbl = yolo100 / 'labels' / 'train' / f'{un}.txt'
            img = Image.open(str(ip)).convert('RGB')
            w, h = img.size
            pad_frac = random.uniform(0.30, 0.50)
            pad_x, pad_y = int(w * pad_frac), int(h * pad_frac)
            bg = tuple(random.randint(170, 240) for _ in range(3))
            padded = ImageOps.expand(img, border=(pad_x, pad_y, pad_x, pad_y), fill=bg)
            padded.save(str(dst_img), quality=90)
            new_w, new_h = w + 2*pad_x, h + 2*pad_y
            cx = (pad_x + w/2) / new_w
            cy = (pad_y + h/2) / new_h
            dst_lbl.write_text(f'{cat_id} {cx:.6f} {cy:.6f} {w/new_w:.6f} {h/new_h:.6f}')
            class_count[cat_id] = class_count.get(cat_id, 0) + 1
            n_prod += 1

    # Val = symlink til train
    if not (yolo100 / 'images' / 'val').exists():
        os.symlink(str(yolo100 / 'images' / 'train'), str(yolo100 / 'images' / 'val'))
        os.symlink(str(yolo100 / 'labels' / 'train'), str(yolo100 / 'labels' / 'val'))

    cats_sorted = sorted(coco['categories'], key=lambda c: c['id'])
    nc = len(cats_sorted)
    names_lines = [f"  {c['id']}: '{c['name'].replace(chr(39), '').replace(chr(34), '')}'"
                   for c in cats_sorted]
    (yolo100 / 'data.yaml').write_text(
        f'path: /content/yolo_100pct\ntrain: images/train\nval: images/val\nnc: {nc}\nnames:\n'
        + '\n'.join(names_lines)
    )
    total = len(list((yolo100 / 'images' / 'train').iterdir()))
    print(f'100%-dataset: {total} bilder ({len(coco["images"])} hylle + {n_prod} produkt)')

    BATCH = 8 if torch.cuda.get_device_properties(0).total_memory > 30e9 else 4
    from ultralytics import YOLO
    model100 = YOLO('yolov8x.pt')
    model100.train(
        data=str(yolo100 / 'data.yaml'),
        epochs=150, imgsz=1280, batch=BATCH, patience=30,
        device=0, project='/content/runs', name='yolov8x_100pct',
        exist_ok=True, save=True, amp=True, workers=2,
        hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
        degrees=3.0, translate=0.1, scale=0.5,
        mosaic=1.0, mixup=0.05, copy_paste=0.3,
        erasing=0.2, fliplr=0.5, close_mosaic=15,
        optimizer='AdamW', lr0=0.001, lrf=0.01,
        warmup_epochs=3, weight_decay=0.0005,
    )
    shutil.copy('/content/runs/yolov8x_100pct/weights/best.pt',
                '/content/drive/MyDrive/yolov8x_100pct_best.pt')
    print('100%-modell lagret til Drive: yolov8x_100pct_best.pt')

In [ ]:
# CELLE 7: Eksporter til ONNX (FP16) og bygg submission
# Velger 85%-modellen eller 100%-modellen automatisk
import json, shutil, torch, functools
from pathlib import Path
from ultralytics import YOLO

torch.load = functools.partial(torch.load, weights_only=False)

with open('/content/val_score.json') as f:
    vs = json.load(f)

best_conf = vs['best_conf']

# Bruk 100%-modellen hvis tilgjengelig, ellers 85%-modellen
pt_100 = Path('/content/runs/yolov8x_100pct/weights/best.pt')
pt_85  = Path('/content/runs/yolov8x_final/weights/best.pt')
pt_src = pt_100 if pt_100.exists() else pt_85
print(f'Eksporterer: {pt_src}')

model = YOLO(str(pt_src))

# FP16 ONNX — docs anbefaler dette for L4 GPU: "smaller and faster"
# opset=17 matcher sandbox (onnxruntime 1.20.0 støtter opp til opset 20)
model.export(
    format='onnx',
    imgsz=960,
    opset=17,
    dynamic=False,
    half=True,   # FP16: anbefalt av docs for L4 GPU — raskere OG mindre fil
)

# ONNX-fil havner ved siden av .pt-filen
onnx_src = pt_src.with_suffix('.onnx')
assert onnx_src.exists(), f'ONNX ikke funnet: {onnx_src}'

# Bygg submission-mappe
sub = Path('/content/submission')
if sub.exists():
    shutil.rmtree(str(sub))
sub.mkdir()

shutil.copy(str(onnx_src), str(sub / 'best.onnx'))
shutil.copy(str(onnx_src), str(Path('/content/drive/MyDrive/yolov8x_final_best_fp16.onnx')))

onnx_mb = (sub / 'best.onnx').stat().st_size / 1024 / 1024
print(f'best.onnx (FP16): {onnx_mb:.1f} MB')
assert onnx_mb < 420, f'ONNX for stor: {onnx_mb:.1f} MB'

# run.py — identisk struktur som 0.8654-submission
# Kun tillatte imports: argparse, json, pathlib, ultralytics
# Ingen: os, sys, subprocess, socket, pickle, shutil, yaml, threading, multiprocessing, gc, etc.
run_py = f'''import argparse
import json
from pathlib import Path
from ultralytics import YOLO


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--input", required=True)
    parser.add_argument("--output", required=True)
    args = parser.parse_args()

    model = YOLO(str(Path(__file__).parent / "best.onnx"))

    img_paths = sorted([
        p for p in Path(args.input).iterdir()
        if p.suffix.lower() in (".jpg", ".jpeg", ".png")
    ])
    print(f"Processing {{len(img_paths)}} images...")

    predictions = []
    for img_path in img_paths:
        image_id = int(img_path.stem.split("_")[-1])
        results = model.predict(
            str(img_path),
            device=0,       # L4 GPU alltid tilgjengelig i sandbox
            conf={best_conf},
            iou=0.45,
            verbose=False,
            imgsz=960,
            augment=True,   # Lett TTA (flip) — rask og gir +1-2% mAP
            half=True,      # FP16 inference — raskere på L4
        )
        for result in results:
            boxes = result.boxes
            for i in range(len(boxes)):
                x1, y1, x2, y2 = boxes.xyxy[i].tolist()
                predictions.append({{
                    "image_id": image_id,
                    "category_id": int(boxes.cls[i].item()),
                    "bbox": [round(x1, 1), round(y1, 1),
                             round(x2 - x1, 1), round(y2 - y1, 1)],
                    "score": round(float(boxes.conf[i].item()), 3),
                }})

    Path(args.output).parent.mkdir(parents=True, exist_ok=True)
    with open(args.output, "w") as f:
        json.dump(predictions, f)
    print(f"Wrote {{len(predictions)}} predictions")


if __name__ == "__main__":
    main()
'''
(sub / 'run.py').write_text(run_py)
print(f'run.py lagret  (conf={best_conf})')

# Fullstendig sjekk av ALLE blokkerte imports fra sandbox-dokumentasjonen
BLOCKED_IMPORTS = [
    'import os', 'import sys', 'import subprocess', 'import socket',
    'import ctypes', 'import builtins', 'import importlib',
    'import pickle', 'import marshal', 'import shelve', 'import shutil',
    'import yaml', 'import requests', 'import urllib', 'import http',
    'import multiprocessing', 'import threading', 'import signal', 'import gc',
    'import code', 'import codeop', 'import pty',
]
BLOCKED_CALLS = ['eval(', 'exec(', 'compile(', '__import__(']

run_content = (sub / 'run.py').read_text()
for b in BLOCKED_IMPORTS + BLOCKED_CALLS:
    assert b not in run_content, f'BLOKKERT I run.py: {b}'
print('Sikkerhetssjekk OK — ingen blokkerte imports eller kall!')

In [ ]:
# CELLE 8: Valider run.py lokalt, lag ZIP og last ned
import subprocess, sys, json, zipfile
from pathlib import Path

sub = Path('/content/submission')

# Test run.py mot val-bilder
pred_path = Path('/content/test_predictions.json')
result = subprocess.run(
    [sys.executable, str(sub / 'run.py'),
     '--input', '/content/val_input',
     '--output', str(pred_path)],
    capture_output=True, text=True, timeout=600
)
print(result.stdout[-1000:])
if result.returncode != 0:
    print('FEIL:', result.stderr[-2000:])
    raise RuntimeError('run.py feilet!')

with open(pred_path) as f:
    preds = json.load(f)
print(f'Prediksjoner OK: {len(preds)} totalt')

# ZIP
weight_exts = {'.pt', '.pth', '.onnx', '.safetensors', '.npy'}
files = sorted(sub.iterdir())
weight_count = sum(1 for f in files if f.suffix in weight_exts)
total_weight_mb = sum(f.stat().st_size for f in files if f.suffix in weight_exts) / 1024 / 1024

print(f'\nFiler i submission:')
for f in files:
    tag = ' [WEIGHT]' if f.suffix in weight_exts else ''
    print(f'  {f.name}: {f.stat().st_size/1024/1024:.1f} MB{tag}')

print(f'\nWeight-filer: {weight_count}/3')
print(f'Total weight: {total_weight_mb:.1f}/420 MB')
assert weight_count <= 3
assert total_weight_mb < 420

zip_path = '/content/submission_mathea_final.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in files:
        zf.write(str(f), f.name)

zip_mb = Path(zip_path).stat().st_size / 1024 / 1024
print(f'ZIP: {zip_mb:.1f} MB')
assert zip_mb < 420

from google.colab import files
files.download(zip_path)
print(f'\nLast opp submission_mathea_final.zip på app.ainm.no/submit/norgesgruppen-data')